# GHIA — Ferramental Quantitativo | Fase 0: Infraestrutura de Coleta de Dados

Protótipo de coleta e tratamento das séries macro e de mercado que servem de base para as fases
seguintes do projeto (faixa de erro histórico do Focus, distribuição condicional de retorno por
classe de ativo, Black-Litterman).

**Fontes:** SGS/Banco Central, SIDRA/IBGE, Boletim Focus (BCB) e IPEADATA.

**Saída:** base mensal única em Parquet, com log de execução e checagem de consistência.

In [1]:
from bcb import sgs
import sidrapy
import ipeadatapy as ipea
import requests
import pandas as pd
import numpy as np
import pyarrow  # engine de parquet — se faltar, instale com `pip install pyarrow` antes de continuar
from datetime import datetime
import json
import logging
import time
import inspect
from pathlib import Path

pd.set_option('display.width', 120)

## 0. Configuração geral (pastas, log, parâmetros)

In [2]:
# Pastas de saída. DATA_DIR guarda a base tratada; RAW_DIR guarda o dado bruto de cada fonte
# (útil para auditar uma coleta que deu errado sem precisar buscar tudo de novo).

DATA_DIR = Path("dados_ghia")
RAW_DIR = DATA_DIR / "raw"
DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

LOG_PATH = DATA_DIR / "log_atualizacao.jsonl"
DATA_INICIO = "2003-01-01"  # início da série histórica

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("ghia_coleta")


def registrar_log(etapa: str, status: str, detalhes: dict | None = None) -> None:
    """
    Registra uma linha de log estruturado (JSON Lines) com o resultado de uma etapa de coleta.

    `status` deve ser "ok" ou "erro". O arquivo acumula um histórico de execuções, permitindo
    checar depois quando cada fonte foi atualizada com sucesso pela última vez.
    """
    registro = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "etapa": etapa,
        "status": status,
        "detalhes": detalhes or {},
    }
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")
    nivel = logging.INFO if status == "ok" else logging.WARNING
    logger.log(nivel, f"[{etapa}] {status} | {detalhes or ''}")


def salvar_parquet_seguro(df: pd.DataFrame, caminho: Path) -> None:
    """
    Salva um DataFrame em parquet sem derrubar a coleta se isso falhar. O parquet bruto
    (pasta `raw/`) é só um artefato de auditoria — a ausência dele não pode custar os
    dados que já foram coletados com sucesso, por isso o erro aqui vira só um aviso no
    log em vez de propagar e ser pego pelo try/except da coleta (que descartaria o
    DataFrame inteiro por causa de uma falha na gravação, não na coleta em si).
    """
    try:
        df.to_parquet(caminho)
    except Exception as e:
        registrar_log("salvar_parquet_raw", "erro", {"caminho": str(caminho), "mensagem": str(e)})


def executar_com_retentativas(func, *args, tentativas: int = 4, espera_inicial: int = 2, **kwargs):
    """
    Executa `func(*args, **kwargs)` com retentativas e backoff exponencial (2s, 4s, 8s,
    16s...). Usado em toda chamada de rede da coleta: uma falha transitória (timeout,
    conexão instável, rede corporativa mais lenta) não deve derrubar a coleta inteira
    na primeira tentativa.
    """
    for tentativa in range(tentativas):
        try:
            return func(*args, **kwargs)
        except Exception:
            if tentativa == tentativas - 1:
                raise
            time.sleep(espera_inicial * (2 ** tentativa))

## 1. Coleta — SGS (Banco Central)

Séries diárias e mensais do Sistema Gerenciador de Séries Temporais. Códigos conferidos
diretamente na API (`api.bcb.gov.br`) em 2026-08-29.

In [3]:
# (código, é_diária) — o SGS limita consultas de série diária a janelas de no máximo
# 10 anos; séries mensais não têm esse limite.
CODIGOS_SGS = {
    "selic_meta": (432, True),              # Meta Selic definida pelo Copom (% a.a.), diária
    "selic_over_mensal": (4390, False),     # Selic acumulada no mês, anualizada (% a.a.)
    "ipca_mensal": (433, False),            # IPCA - variação mensal (%)
    "ipca_12m": (13522, False),             # IPCA - variação acumulada em 12 meses (%)
    "cambio_compra": (1, True),             # Dólar americano (compra), câmbio livre, diário
    "ibcbr_dessaz": (24364, False),         # IBC-Br, com ajuste sazonal
    "credito_saldo_total": (20539, False),  # Saldo da carteira de crédito - total (R$ milhões)
    "cdi_mensal": (4391, False),            # CDI acumulado no mês (%) — usado como retorno da classe CDI na Fase 2
}


def _dividir_em_janelas(inicio: pd.Timestamp, fim: pd.Timestamp, anos: int = 9) -> list[tuple]:
    """Quebra um intervalo de datas em janelas de até `anos` anos (fecho à direita inclusive)."""
    janelas = []
    cursor = inicio
    while cursor < fim:
        fim_janela = min(cursor + pd.DateOffset(years=anos, days=-1), fim)
        janelas.append((cursor, fim_janela))
        cursor = fim_janela + pd.DateOffset(days=1)
    return janelas


SGS_SUPORTA_TIMEOUT = "timeout" in inspect.signature(sgs.get).parameters


def _sgs_get(codigos: dict, start: str, end: str) -> pd.DataFrame:
    """
    Chama `sgs.get`, passando `timeout` só se a versão instalada do python-bcb suportar
    o parâmetro — versões mais antigas da biblioteca não têm esse argumento e quebram
    com TypeError se ele for passado.
    """
    if SGS_SUPORTA_TIMEOUT:
        return sgs.get(codigos, start=start, end=end, timeout=60)
    return sgs.get(codigos, start=start, end=end)


def _coletar_serie_sgs(nome: str, codigo: int, data_inicio: str, diaria: bool) -> pd.DataFrame:
    """
    Coleta uma única série do SGS. Uma série por chamada (em vez de várias num só
    `sgs.get`) — testado e mais estável que pedir várias séries de uma vez, que se
    mostrou instável nesta rede. Séries diárias são buscadas em janelas de até 9 anos
    para respeitar o limite de 10 anos da API. Timeout generoso (60s, quando suportado)
    e retentativas: numa conexão mais lenta (rede corporativa, VPN) o timeout padrão da
    biblioteca estoura antes de a API do BCB responder.
    """
    inicio, fim = pd.Timestamp(data_inicio), pd.Timestamp.today()
    janelas = _dividir_em_janelas(inicio, fim) if diaria else [(inicio, fim)]
    partes = [
        executar_com_retentativas(_sgs_get, {nome: codigo}, str(i.date()), str(f.date()))
        for i, f in janelas
    ]
    serie = pd.concat(partes).sort_index()
    return serie[~serie.index.duplicated(keep="last")]


def coletar_sgs(codigos: dict, data_inicio: str = DATA_INICIO) -> pd.DataFrame:
    """
    Coleta múltiplas séries do SGS/Banco Central e retorna um único DataFrame indexado
    por data (uma coluna por série). Séries de frequências diferentes (diária, mensal)
    ficam com NaN nas datas em que não há observação — o alinhamento de calendário é
    responsabilidade da etapa de tratamento, não da coleta.
    """
    colunas = [_coletar_serie_sgs(nome, codigo, data_inicio, diaria) for nome, (codigo, diaria) in codigos.items()]
    df = pd.concat(colunas, axis=1)
    df.index.name = "data"
    return df


try:
    df_sgs = coletar_sgs(CODIGOS_SGS)
    salvar_parquet_seguro(df_sgs, RAW_DIR / "sgs.parquet")
    registrar_log("coleta_sgs", "ok", {"n_series": len(CODIGOS_SGS), "n_obs": len(df_sgs)})
except Exception as e:
    df_sgs = pd.DataFrame()
    registrar_log("coleta_sgs", "erro", {"mensagem": str(e)})

df_sgs.tail()

2026-09-01 00:52:44,038 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=31%2F12%2F2011 "HTTP/1.1 200 OK"


2026-09-01 00:52:44,984 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json&dataInicial=01%2F01%2F2012&dataFinal=31%2F12%2F2020 "HTTP/1.1 200 OK"


2026-09-01 00:52:45,606 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json&dataInicial=01%2F01%2F2021&dataFinal=01%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-01 00:52:45,882 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.4390/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=01%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-01 00:52:46,123 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=01%2F09%2F2026 "HTTP/1.1 200 OK"


...

2026-09-01 00:52:47,529 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.24364/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=01%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-01 00:52:47,764 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.20539/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=01%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-01 00:52:47,998 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.4391/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=01%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-01 00:52:48,072 | INFO | [coleta_sgs] ok | {'n_series': 8, 'n_obs': 8645}


,selic_meta,selic_over_mensal,ipca_mensal,ipca_12m,cambio_compra,ibcbr_dessaz,credito_saldo_total,cdi_mensal
data,,,,,,,,
2026-08-28,14.0,NaN,NaN,NaN,5.2005,NaN,NaN,NaN
2026-08-29,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-30,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-31,14.0,NaN,NaN,NaN,5.1816,NaN,NaN,NaN
2026-09-01,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Coleta — SIDRA (IBGE)

Taxa de desocupação da PNAD Contínua (trimestre móvel), tabela 6381, agregado Brasil.

In [4]:
def coletar_sidra_desocupacao() -> pd.DataFrame:
    """
    Coleta a série completa da taxa de desocupação (PNAD Contínua, trimestre móvel,
    Brasil) via SIDRA e retorna um DataFrame mensal com a taxa em pontos percentuais.
    A data atribuída a cada trimestre móvel é o último mês do trimestre.
    """
    bruto = executar_com_retentativas(
        sidrapy.get_table,
        table_code="6381",
        territorial_level="1",
        ibge_territorial_code="all",
        variable="4099",
        period="all",
    )
    bruto = bruto.iloc[1:].copy()  # primeira linha é o cabeçalho descritivo (D2C = Trimestre Móvel)
    bruto["data"] = pd.to_datetime(bruto["D2C"], format="%Y%m")
    bruto["taxa_desocupacao"] = pd.to_numeric(bruto["V"], errors="coerce")
    return bruto.set_index("data")[["taxa_desocupacao"]].sort_index()


try:
    df_sidra = coletar_sidra_desocupacao()
    salvar_parquet_seguro(df_sidra, RAW_DIR / "sidra_desocupacao.parquet")
    registrar_log("coleta_sidra", "ok", {"n_obs": len(df_sidra)})
except Exception as e:
    df_sidra = pd.DataFrame()
    registrar_log("coleta_sidra", "erro", {"mensagem": str(e)})

df_sidra.tail()

2026-09-01 00:52:49,577 | INFO | [coleta_sidra] ok | {'n_obs': 173}


,taxa_desocupacao
data,
2026-03-01,6.1
2026-04-01,5.8
2026-05-01,5.6
2026-06-01,5.4
2026-07-01,5.3


## 3. Coleta — Boletim Focus (expectativas de mercado, BCB)

Serviço OData `Expectativas` do BCB (Olinda). Duas séries relevantes para a Fase 1
(faixa de erro histórico do Focus):

- `ExpectativaMercadoMensais`: mediana/média por indicador e mês de referência.
- `ExpectativasMercadoInflacao12Meses`: mediana/média da inflação acumulada nos
  próximos 12 meses — é a série de horizonte fixo que permite medir erro por horizonte
  sem ter que reconstruir o "horizonte" a partir da data de referência.

In [5]:
FOCUS_BASE_URL = "https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata"


def _consultar_focus_odata(entidade: str, filtro: str, orderby: str = "Data") -> pd.DataFrame:
    """
    Faz uma consulta genérica ao serviço OData de Expectativas do BCB, paginando em
    blocos de 1000 registros (limite do serviço) até esgotar o resultado.
    """
    registros = []
    skip = 0
    tamanho_pagina = 1000
    while True:
        url = (
            f"{FOCUS_BASE_URL}/{entidade}"
            f"?$filter={filtro}&$orderby={orderby}&$format=json"
            f"&$top={tamanho_pagina}&$skip={skip}"
        )
        resp = executar_com_retentativas(requests.get, url, timeout=60)
        resp.raise_for_status()
        pagina = resp.json()["value"]
        registros.extend(pagina)
        if len(pagina) < tamanho_pagina:
            break
        skip += tamanho_pagina
    return pd.DataFrame(registros)


def coletar_focus_ipca_12m(suavizada: str = "N") -> pd.DataFrame:
    """
    Coleta a série de expectativa de IPCA acumulado em 12 meses à frente (horizonte fixo),
    versão não suavizada por padrão. Retorna média, mediana e desvio-padrão por data de
    coleta, indexado por data.
    """
    # baseCalculo=0 ("todas as coletas do dia") garante uma linha por data; baseCalculo=1
    # é uma variante intradiária (coletas antes do horário de corte) que duplicaria o índice.
    filtro = f"Indicador eq 'IPCA' and Suavizada eq '{suavizada}' and baseCalculo eq 0"
    df = _consultar_focus_odata("ExpectativasMercadoInflacao12Meses", filtro)
    df["Data"] = pd.to_datetime(df["Data"])
    return df.set_index("Data").sort_index()


def coletar_focus_mensal(indicador: str = "IPCA", anos_recentes: int = 2) -> pd.DataFrame:
    """
    Coleta a série de expectativas mensais (por mês de referência) para um indicador do
    Focus (ex.: IPCA, Câmbio, Selic). Cada linha é uma combinação (data de coleta, mês de
    referência) — granularidade fina, útil para horizontes diferentes de 12 meses.

    Limitada aos últimos `anos_recentes` anos por desempenho: essa tabela tem frequência
    diária cruzada com dezenas de meses de referência, e o serviço do BCB pagina devagar
    em consultas longas (~1.5s por 1000 linhas). Para o histórico completo em produção,
    rodar esta função como job em lote separado, fora deste notebook interativo.
    """
    data_minima = (pd.Timestamp.today() - pd.DateOffset(years=anos_recentes)).strftime("%Y-%m-%d")
    filtro = f"Indicador eq '{indicador}' and Data ge '{data_minima}'"
    df = _consultar_focus_odata("ExpectativaMercadoMensais", filtro)
    df["Data"] = pd.to_datetime(df["Data"])
    df["DataReferencia"] = pd.to_datetime(df["DataReferencia"], format="%m/%Y")
    return df.sort_values(["DataReferencia", "Data"])


try:
    df_focus_12m = coletar_focus_ipca_12m()
    salvar_parquet_seguro(df_focus_12m, RAW_DIR / "focus_ipca_12m.parquet")
    registrar_log("coleta_focus_12m", "ok", {"n_obs": len(df_focus_12m)})
except Exception as e:
    df_focus_12m = pd.DataFrame()
    registrar_log("coleta_focus_12m", "erro", {"mensagem": str(e)})

try:
    df_focus_ipca_mensal = coletar_focus_mensal("IPCA")
    salvar_parquet_seguro(df_focus_ipca_mensal, RAW_DIR / "focus_ipca_mensal.parquet")
    registrar_log("coleta_focus_mensal", "ok", {"n_obs": len(df_focus_ipca_mensal)})
except Exception as e:
    df_focus_ipca_mensal = pd.DataFrame()
    registrar_log("coleta_focus_mensal", "erro", {"mensagem": str(e)})

df_focus_12m.tail()

2026-09-01 00:52:59,676 | INFO | [coleta_focus_12m] ok | {'n_obs': 6227}


2026-09-01 00:53:37,440 | INFO | [coleta_focus_mensal] ok | {'n_obs': 25050}


,Indicador,Suavizada,Media,Mediana,DesvioPadrao,Minimo,Maximo,numeroRespondentes,baseCalculo
Data,,,,,,,,,
2026-08-24,IPCA,N,4.2922,4.3160,0.4970,2.694,5.9637,132.0,0
2026-08-25,IPCA,N,4.3003,4.3413,0.4978,2.694,5.9637,133.0,0
2026-08-26,IPCA,N,4.2919,4.3209,0.5035,2.694,5.9637,133.0,0
2026-08-27,IPCA,N,4.2971,4.3455,0.5079,2.694,5.9637,133.0,0
2026-08-28,IPCA,N,4.2964,4.3455,0.5118,2.694,5.9637,133.0,0


## 4. Coleta — IPEADATA

Fonte adicional mencionada no briefing. A API pública do IPEADATA é historicamente
instável (fora do ar com frequência) — por isso a coleta é isolada em try/except e não
derruba o pipeline caso falhe; a base tratada final simplesmente sai sem essas colunas
naquela execução, e o log registra o problema para investigar depois.

In [6]:
CODIGOS_IPEADATA = {
    "ipca_ipea_12m": "PRECOS12_IPCAGA12",      # IPCA, var. acumulada 12 meses (%) — cruzamento com SGS
    "expectativa_ipca_ipea": "BM12_IPCAEXP1212",  # Expectativa média de inflação 12 meses (%)
}


def coletar_ipeadata(codigos: dict) -> pd.DataFrame:
    """
    Coleta múltiplas séries do IPEADATA e as consolida em um único DataFrame mensal.
    Mantém apenas a última coluna de cada série retornada pela biblioteca (valor numérico),
    descartando metadados redundantes.
    """
    colunas = []
    for nome, codigo in codigos.items():
        serie = ipea.timeseries(codigo).iloc[:, [-1]]
        serie.columns = [nome]
        colunas.append(serie)
    return pd.concat(colunas, axis=1)


try:
    df_ipea = coletar_ipeadata(CODIGOS_IPEADATA)
    salvar_parquet_seguro(df_ipea, RAW_DIR / "ipeadata.parquet")
    registrar_log("coleta_ipeadata", "ok", {"n_series": len(CODIGOS_IPEADATA), "n_obs": len(df_ipea)})
except Exception as e:
    df_ipea = pd.DataFrame()
    registrar_log("coleta_ipeadata", "erro", {"mensagem": str(e)})

df_ipea.tail()

2026-09-01 00:53:40,902 | INFO | [coleta_ipeadata] ok | {'n_series': 2, 'n_obs': 548}


,ipca_ipea_12m,expectativa_ipca_ipea
DATE,,
2026-03-01,4.14,4.2281
2026-04-01,4.39,4.2360
2026-05-01,4.72,4.1949
2026-06-01,4.64,4.2177
2026-07-01,4.44,4.0830


## 5. Tratamento — consolidação em base mensal única

Regra de alinhamento: séries diárias (Selic meta, câmbio) viram média mensal;
séries que já nascem mensais são apenas reindexadas para o primeiro dia do mês.
Nenhuma dessazonalização é aplicada aqui — o IBC-Br e a taxa de desocupação já vêm
dessazonalizados na fonte; séries que precisarem de ajuste específico por classe de
ativo entram nessa etapa nas fases seguintes, não na infraestrutura de coleta.

In [7]:
SERIES_DIARIAS = [nome for nome, (_, diaria) in CODIGOS_SGS.items() if diaria]


def mensualizar(df: pd.DataFrame, colunas_diarias: list[str]) -> pd.DataFrame:
    """
    Recebe um DataFrame com índice de data (frequência mista) e devolve uma versão
    mensal: colunas em `colunas_diarias` são agregadas pela média do mês; as demais
    colunas são reamostradas por último valor não nulo do mês (já são mensais na origem).
    """
    diarias = df[colunas_diarias].resample("MS").mean()
    mensais = df.drop(columns=colunas_diarias).resample("MS").last()
    return diarias.join(mensais, how="outer")


df_sgs_mensal = mensualizar(df_sgs, SERIES_DIARIAS) if not df_sgs.empty else pd.DataFrame()
df_sidra_mensal = df_sidra.resample("MS").last() if not df_sidra.empty else pd.DataFrame()

base = df_sgs_mensal.join(df_sidra_mensal, how="outer")
if not df_ipea.empty:
    df_ipea.index = pd.to_datetime(df_ipea.index)
    base = base.join(df_ipea.resample("MS").last(), how="outer")

base = base.sort_index()
base.tail()

,selic_meta,cambio_compra,selic_over_mensal,ipca_mensal,ipca_12m,ibcbr_dessaz,credito_saldo_total,cdi_mensal,taxa_desocupacao,ipca_ipea_12m,expectativa_ipca_ipea
data,,,,,,,,,,,
2026-05-01,14.500000,4.983700,1.07,0.58,4.72,110.93102,7305306.0,1.07,5.6,4.72,4.1949
2026-06-01,14.391667,5.127571,1.12,0.16,4.64,110.22154,7353293.0,1.12,5.4,4.64,4.2177
2026-07-01,14.250000,5.113948,1.22,0.07,4.44,NaN,7372243.0,1.22,5.3,4.44,4.0830
2026-08-01,14.040323,5.153162,1.09,NaN,NaN,NaN,NaN,1.04,NaN,NaN,NaN
2026-09-01,14.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Checagem de consistência

In [8]:
def checar_consistencia(df: pd.DataFrame) -> dict:
    """
    Roda checagens básicas de qualidade sobre a base consolidada: percentual de nulos
    por coluna, existência de datas duplicadas e maior gap (em meses) sem nenhuma
    observação registrada em pelo menos uma série. Não corrige nada — só relata.

    Se `df` vier vazio (nenhuma linha), é sinal de que uma das coletas anteriores
    falhou silenciosamente — os try/except de cada fonte evitam que o pipeline pare,
    mas isso pode mascarar o problema até aqui. Nesse caso, a função não tenta calcular
    período/gaps (não há o que calcular) e devolve um relatório sinalizando a falha, em
    vez de estourar um erro genérico do pandas ao comparar datas com NaT.
    """
    if df.empty:
        return {
            "erro": "base vazia — verifique os logs de coleta acima (SGS, SIDRA, Focus, IPEADATA) para achar qual fonte falhou",
            "n_linhas": 0,
        }

    duplicadas = int(df.index.duplicated().sum())
    nulos_pct = (df.isna().mean() * 100).round(1).to_dict()

    calendario_completo = pd.date_range(df.index.min(), df.index.max(), freq="MS")
    meses_faltantes = calendario_completo.difference(df.index)

    return {
        "periodo": [str(df.index.min().date()), str(df.index.max().date())],
        "n_linhas": len(df),
        "datas_duplicadas": duplicadas,
        "nulos_pct_por_coluna": nulos_pct,
        "meses_faltantes_no_indice": [str(d.date()) for d in meses_faltantes],
    }


relatorio = checar_consistencia(base)
registrar_log("checagem_consistencia", "ok", relatorio)
relatorio

2026-09-01 00:53:40,950 | INFO | [checagem_consistencia] ok | {'periodo': ['1980-12-01', '2026-09-01'], 'n_linhas': 550, 'datas_duplicadas': 0, 'nulos_pct_por_coluna': {'selic_meta': 48.2, 'cambio_compra': 48.4, 'selic_over_mensal': 48.4, 'ipca_mensal': 48.5, 'ipca_12m': 48.5, 'ibcbr_dessaz': 48.7, 'credito_saldo_total': 48.5, 'cdi_mensal': 48.4, 'taxa_desocupacao': 68.5, 'ipca_ipea_12m': 0.4, 'expectativa_ipca_ipea': 45.3}, 'meses_faltantes_no_indice': []}


{'periodo': ['1980-12-01', '2026-09-01'],
 'n_linhas': 550,
 'datas_duplicadas': 0,
 'nulos_pct_por_coluna': {'selic_meta': 48.2,
  'cambio_compra': 48.4,
  'selic_over_mensal': 48.4,
  'ipca_mensal': 48.5,
  'ipca_12m': 48.5,
  'ibcbr_dessaz': 48.7,
  'credito_saldo_total': 48.5,
  'cdi_mensal': 48.4,
  'taxa_desocupacao': 68.5,
  'ipca_ipea_12m': 0.4,
  'expectativa_ipca_ipea': 45.3},
 'meses_faltantes_no_indice': []}

## 7. Exportação da base tratada

In [9]:
CAMINHO_BASE = DATA_DIR / "base_ghia_mensal.parquet"

base.to_parquet(CAMINHO_BASE)
registrar_log("exportacao_base", "ok", {"caminho": str(CAMINHO_BASE), "n_linhas": len(base), "n_colunas": base.shape[1]})

print(f"Base exportada: {CAMINHO_BASE} ({base.shape[0]} linhas x {base.shape[1]} colunas)")
base.tail()

2026-09-01 00:53:40,961 | INFO | [exportacao_base] ok | {'caminho': 'dados_ghia/base_ghia_mensal.parquet', 'n_linhas': 550, 'n_colunas': 11}


Base exportada: dados_ghia/base_ghia_mensal.parquet (550 linhas x 11 colunas)


,selic_meta,cambio_compra,selic_over_mensal,ipca_mensal,ipca_12m,ibcbr_dessaz,credito_saldo_total,cdi_mensal,taxa_desocupacao,ipca_ipea_12m,expectativa_ipca_ipea
data,,,,,,,,,,,
2026-05-01,14.500000,4.983700,1.07,0.58,4.72,110.93102,7305306.0,1.07,5.6,4.72,4.1949
2026-06-01,14.391667,5.127571,1.12,0.16,4.64,110.22154,7353293.0,1.12,5.4,4.64,4.2177
2026-07-01,14.250000,5.113948,1.22,0.07,4.44,NaN,7372243.0,1.22,5.3,4.44,4.0830
2026-08-01,14.040323,5.153162,1.09,NaN,NaN,NaN,NaN,1.04,NaN,NaN,NaN
2026-09-01,14.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---

# Fase 1 — Faixa de erro histórico do Focus (IPCA), por horizonte

Não é um modelo novo, é honestidade estatística: comparar a mediana do Focus, coletada
todo dia, com a inflação que de fato aconteceu nos meses seguintes. O objetivo é uma
frase do tipo *"a projeção mediana para o IPCA é X, e historicamente a previsão a doze
meses erra em mais ou menos Y pontos em 70% dos casos"* — sem tentar bater o Focus.

## 1.1 Coleta — expectativa de IPCA a 24 meses

A série de 12 meses (`df_focus_12m`) já foi coletada na Fase 0, com histórico desde 2001.
A de 24 meses só existe desde 2021 no serviço do BCB — entra como segundo horizonte de
comparação, com a ressalva de amostra mais curta.

In [10]:
def coletar_focus_inflacao_horizonte(entidade: str, indicador: str = "IPCA", suavizada: str = "N") -> pd.DataFrame:
    """
    Coleta uma série de expectativa de inflação de horizonte fixo do Focus (12 ou 24
    meses à frente, conforme `entidade`). Filtra baseCalculo=0 para uma linha por data.
    """
    filtro = f"Indicador eq '{indicador}' and Suavizada eq '{suavizada}' and baseCalculo eq 0"
    df = _consultar_focus_odata(entidade, filtro)
    df["Data"] = pd.to_datetime(df["Data"])
    return df.set_index("Data").sort_index()


try:
    df_focus_24m = coletar_focus_inflacao_horizonte("ExpectativasMercadoInflacao24Meses")
    salvar_parquet_seguro(df_focus_24m, RAW_DIR / "focus_ipca_24m.parquet")
    registrar_log("coleta_focus_24m", "ok", {"n_obs": len(df_focus_24m)})
except Exception as e:
    df_focus_24m = pd.DataFrame()
    registrar_log("coleta_focus_24m", "erro", {"mensagem": str(e)})

df_focus_24m[["Mediana"]].describe()

2026-09-01 00:53:43,784 | INFO | [coleta_focus_24m] ok | {'n_obs': 1361}


,Mediana
count,1361.000000
mean,3.849846
std,0.237710
min,3.335000
25%,3.669700
50%,3.801600
75%,3.991100
max,4.457200


## 1.2 Índice de preços acumulado (a partir do IPCA mensal)

Para saber quanto o IPCA realmente acumulou entre o mês da coleta e `h` meses depois,
construímos um índice de preços a partir da variação mensal do IPCA (já coletada na
Fase 0). A inflação realizada num intervalo é simplesmente a razão entre o índice no
fim e no início do intervalo.

In [11]:
def construir_indice_precos(ipca_mensal: pd.Series) -> pd.Series:
    """
    Constrói um índice de preços (base 100 no primeiro mês) a partir de uma série de
    variação mensal do IPCA em %. Meses sem dado ainda publicado (cauda da série) são
    descartados antes de acumular, para não interromper o índice no meio.
    """
    serie = ipca_mensal.dropna()
    fatores = 1 + serie / 100
    return 100 * fatores.cumprod()


indice_precos = construir_indice_precos(base["ipca_mensal"])
indice_precos.tail()

data
2026-03-01    369.923948
2026-04-01    372.402439
2026-05-01    374.562373
2026-06-01    375.161673
2026-07-01    375.424286
Freq: MS, Name: ipca_mensal, dtype: float64

## 1.3 Erro do Focus por horizonte (previsto vs. realizado)

Atenção a uma armadilha específica desta série: `ExpectativasMercadoInflacao24Meses` **não**
é a inflação acumulada nos 24 meses seguintes — é a taxa esperada para o *segundo* ano à
frente (meses 12 a 24), sozinho. Confirmado empiricamente: comparar essa mediana contra a
inflação acumulada de 0 a 24 meses gera um erro médio de ~6,7 p.p. (viés absurdo para uma
previsão de mercado); comparando corretamente contra a inflação realizada só entre os meses
12 e 24, o erro médio cai para ~0,8 p.p. — nível compatível com o do horizonte de 12 meses.
Por isso a função abaixo recebe uma janela `(meses_inicio, meses_fim)` em vez de um único
horizonte: o de 12 meses usa (0, 12); o de 24 meses usa (12, 24).

In [12]:
def calcular_erro_focus(previsoes: pd.DataFrame, indice_precos: pd.Series, meses_inicio: int, meses_fim: int) -> pd.DataFrame:
    """
    Para cada data de coleta do Focus, compara a mediana prevista (inflação acumulada
    na janela [meses_inicio, meses_fim) a partir do mês de referência) com a inflação de
    fato realizada na mesma janela, medida pelo índice de preços. Descarta linhas cuja
    janela ainda não se completou (mês final além do último IPCA publicado) — não é
    erro, é previsão que ainda não pôde ser julgada.

    Retorna uma linha por coleta, com previsto, realizado, erro (realizado - previsto:
    positivo quando o Focus subestimou a inflação) e o rótulo do horizonte (meses_fim,
    usado só para identificar/agrupar).
    """
    df = previsoes[["Mediana"]].rename(columns={"Mediana": "previsto"}).copy()
    df["mes_referencia"] = df.index.to_period("M").to_timestamp()
    mes_inicio_janela = df["mes_referencia"] + pd.DateOffset(months=meses_inicio)
    mes_fim_janela = df["mes_referencia"] + pd.DateOffset(months=meses_fim)

    indice_inicio = indice_precos.reindex(mes_inicio_janela).to_numpy()
    indice_fim = indice_precos.reindex(mes_fim_janela).to_numpy()
    df["realizado"] = (indice_fim / indice_inicio - 1) * 100
    df["erro"] = df["realizado"] - df["previsto"]
    df["horizonte_meses"] = meses_fim

    return df.dropna(subset=["realizado"])[
        ["mes_referencia", "previsto", "realizado", "erro", "horizonte_meses"]
    ]


df_erro_12m = calcular_erro_focus(df_focus_12m, indice_precos, meses_inicio=0, meses_fim=12)
df_erro_24m = calcular_erro_focus(df_focus_24m, indice_precos, meses_inicio=12, meses_fim=24)
df_erros_focus = pd.concat([df_erro_12m, df_erro_24m], ignore_index=True)

registrar_log("erro_focus_por_horizonte", "ok", {
    "n_obs_12m": len(df_erro_12m),
    "n_obs_24m": len(df_erro_24m),
})

df_erros_focus.groupby("horizonte_meses")["erro"].describe()

2026-09-01 00:53:43,820 | INFO | [erro_focus_por_horizonte] ok | {'n_obs_12m': 5664, 'n_obs_24m': 838}


,count,mean,std,min,25%,50%,75%,max
horizonte_meses,,,,,,,,
12,5664.0,0.627276,2.139112,-5.732783,-0.657051,0.386637,1.519068,8.244281
24,838.0,0.805636,0.453876,-0.467099,0.477045,0.906083,1.119630,1.515535


## 1.4 Faixa de incerteza — a frase final

`Y70` é o valor tal que, historicamente, o erro absoluto da mediana do Focus ficou
dentro de ± Y70 pontos em 70% das vezes (mesma leitura para 80%). Não é intervalo de
confiança formal — é a estatística mais simples que já responde à pergunta da mesa:
"o Focus está dizendo X, quanto isso pode errar na prática?"

In [13]:
def resumir_incerteza_focus(df_erros: pd.DataFrame, niveis: tuple[float, ...] = (0.70, 0.80, 0.90)) -> pd.DataFrame:
    """
    Resume, por horizonte, a faixa de erro absoluto do Focus nos níveis de confiança
    pedidos (ex.: erro absoluto ficou dentro de ± Y em 70% dos casos históricos).
    """
    linhas = []
    for horizonte, grupo in df_erros.groupby("horizonte_meses"):
        erro_abs = grupo["erro"].abs()
        linha = {
            "horizonte_meses": horizonte,
            "n_obs": len(grupo),
            "erro_medio": grupo["erro"].mean(),
            "erro_absoluto_medio": erro_abs.mean(),
        }
        for nivel in niveis:
            linha[f"faixa_{int(nivel*100)}pct"] = erro_abs.quantile(nivel)
        linhas.append(linha)
    return pd.DataFrame(linhas).set_index("horizonte_meses").round(2)


resumo_incerteza = resumir_incerteza_focus(df_erros_focus)
registrar_log("resumo_incerteza_focus", "ok", resumo_incerteza.to_dict(orient="index"))
resumo_incerteza

2026-09-01 00:53:43,840 | INFO | [resumo_incerteza_focus] ok | {12: {'n_obs': 5664, 'erro_medio': 0.63, 'erro_absoluto_medio': 1.56, 'faixa_70pct': 1.77, 'faixa_80pct': 2.38, 'faixa_90pct': 3.53}, 24: {'n_obs': 838, 'erro_medio': 0.81, 'erro_absoluto_medio': 0.83, 'faixa_70pct': 1.06, 'faixa_80pct': 1.16, 'faixa_90pct': 1.39}}


,n_obs,erro_medio,erro_absoluto_medio,faixa_70pct,faixa_80pct,faixa_90pct
horizonte_meses,,,,,,
12,5664,0.63,1.56,1.77,2.38,3.53
24,838,0.81,0.83,1.06,1.16,1.39


In [14]:
DESCRICAO_HORIZONTE = {
    12: "o IPCA acumulado nos próximos 12 meses",
    24: "o IPCA do segundo ano à frente (entre o 12º e o 24º mês)",
}


def frase_incerteza_focus(horizonte_meses: int, previsoes: pd.DataFrame, resumo: pd.DataFrame, nivel: float = 0.70) -> str:
    """
    Monta a frase de comunicação de incerteza para um horizonte: previsão atual do
    Focus + faixa histórica de erro absoluto no nível de confiança escolhido.
    """
    previsto_atual = previsoes["Mediana"].iloc[-1]
    data_previsao = previsoes.index[-1].strftime("%d/%m/%Y")
    faixa = resumo.loc[horizonte_meses, f"faixa_{int(nivel*100)}pct"]
    descricao = DESCRICAO_HORIZONTE[horizonte_meses]
    return (
        f"Em {data_previsao}, a projeção mediana do Focus para {descricao} é "
        f"{previsto_atual:.2f}%. Historicamente, essa previsão erra em mais ou menos "
        f"{faixa:.2f} pontos percentuais em {int(nivel*100)}% dos casos "
        f"(com base em {int(resumo.loc[horizonte_meses, 'n_obs'])} observações)."
    )


print(frase_incerteza_focus(12, df_focus_12m, resumo_incerteza))
print(frase_incerteza_focus(24, df_focus_24m, resumo_incerteza))

Em 28/08/2026, a projeção mediana do Focus para o IPCA acumulado nos próximos 12 meses é 4.35%. Historicamente, essa previsão erra em mais ou menos 1.77 pontos percentuais em 70% dos casos (com base em 5664 observações).
Em 28/08/2026, a projeção mediana do Focus para o IPCA do segundo ano à frente (entre o 12º e o 24º mês) é 3.94%. Historicamente, essa previsão erra em mais ou menos 1.06 pontos percentuais em 70% dos casos (com base em 838 observações).


---

# Fase 2 — Distribuição condicional de retorno por classe de ativo

Pergunta: dado o cenário atual de Selic e IPCA, qual a distribuição esperada de retorno
para os próximos meses? Usa o Focus como insumo (a Fase 1 já mede quanto confiar nele),
não compete com ele.

**Lacuna de dados conhecida:** as classes IPCA+ longo, prefixado e bolsa local dependem
de uma fonte de cotações que a casa ainda não definiu (o briefing aponta "possivelmente
Comdinheiro"). Testado neste ambiente: Yahoo Finance e Stooq estão inacessíveis (erro de
rede). Por isso a v1 usa apenas **CDI** (SGS) e **dólar** (câmbio, já coletado na Fase 0)
— dado real, não simulado. A metodologia é genérica: adicionar uma nova classe de ativo
mais tarde é só passar a série de retorno para as mesmas funções.

Técnicas, na ordem em que aparecem abaixo:
1. Regressão quantílica (quantis 10/50/90) condicionada ao cenário (nível e variação de
   Selic e IPCA).
2. Conformal prediction (CQR com janela de calibração móvel) para corrigir a cobertura
   do intervalo — validado com backtest em janela expansível, nunca embaralhado.
3. Bootstrap em blocos para simular a distribuição do retorno **acumulado em 12 meses**,
   preservando a dependência temporal dos resíduos.

In [15]:
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg

## 2.1 Cenário (X) e retorno-alvo (y), com alinhamento temporal correto

`X` no mês `t` usa apenas informação já publicada até o fim de `t` (nível de Selic e
IPCA 12m, e suas variações mensais). `y` no mês `t` é o retorno **realizado em `t+1`**
— por isso o `shift(-1)`. Isso garante que, ao treinar com dados até a linha `i`, o
modelo nunca viu informação do futuro em relação ao que está prevendo.

In [16]:
cenario = base[["selic_meta", "ipca_12m"]].copy()
cenario["delta_selic"] = cenario["selic_meta"].diff()
cenario["delta_ipca_12m"] = cenario["ipca_12m"].diff()

retorno_dolar = base["cambio_compra"].pct_change() * 100
retornos = pd.DataFrame({
    "cdi": base["cdi_mensal"],
    "dolar": retorno_dolar,
})

COLUNAS_X = ["selic_meta", "ipca_12m", "delta_selic", "delta_ipca_12m"]
dados_modelo = cenario.join(retornos.shift(-1), how="inner").dropna()

print(f"{len(dados_modelo)} observações mensais utilizáveis, de {dados_modelo.index.min().date()} a {dados_modelo.index.max().date()}")
dados_modelo.tail()

282 observações mensais utilizáveis, de 2003-02-01 a 2026-07-01


,selic_meta,ipca_12m,delta_selic,delta_ipca_12m,cdi,dolar
data,,,,,,
2026-03-01,14.895161,4.14,-0.104839,0.33,1.09,-3.794477
2026-04-01,14.741667,4.39,-0.153495,0.25,1.07,-0.981011
2026-05-01,14.500000,4.72,-0.241667,0.33,1.12,2.886840
2026-06-01,14.391667,4.64,-0.108333,-0.08,1.22,-0.265693
2026-07-01,14.250000,4.44,-0.141667,-0.20,1.04,0.766806


## 2.2 Backtest walk-forward com regressão quantílica + conformal (CQR)

A cada mês do backtest: treina só com o passado (janela expansível), prevê os quantis
10/50/90 para o mês seguinte, e corrige o intervalo [q10, q90] com os erros de
conformidade dos últimos `janela_calibracao` meses (nunca usando o próprio mês testado
— por isso a correção só existe depois que a janela de calibração está cheia).

In [17]:
def backtest_walk_forward_cqr(
    dados: pd.DataFrame,
    colunas_x: list[str],
    coluna_y: str,
    q_low: float = 0.1,
    q_high: float = 0.9,
    janela_minima: int = 120,
    janela_calibracao: int = 36,
) -> pd.DataFrame:
    """
    Backtest de regressão quantílica (quantis q_low/mediana/q_high) em janela
    expansível, com correção conformal (CQR) por janela de calibração móvel.

    Retorna uma linha por mês testado, com a previsão bruta, a previsão corrigida
    pelo conformal e o valor realizado — a base para checar calibração depois.
    """
    dados = dados.sort_index()
    X = sm.add_constant(dados[colunas_x])
    y = dados[coluna_y]
    nivel_cobertura_nominal = q_high - q_low

    registros = []
    scores_conformidade = []

    for i in range(janela_minima, len(dados)):
        X_treino, y_treino = X.iloc[:i], y.iloc[:i]
        X_teste = X.iloc[[i]]
        y_real = y.iloc[i]

        preds = {}
        for q in (q_low, 0.5, q_high):
            resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)
            preds[q] = resultado.predict(X_teste).iloc[0]

        if len(scores_conformidade) >= janela_calibracao:
            correcao = max(np.quantile(scores_conformidade[-janela_calibracao:], nivel_cobertura_nominal), 0.0)
            q_low_cqr, q_high_cqr = preds[q_low] - correcao, preds[q_high] + correcao
        else:
            q_low_cqr, q_high_cqr = np.nan, np.nan

        registros.append({
            "data": dados.index[i],
            "y_real": y_real,
            "q_low_bruto": preds[q_low],
            "q_mediana": preds[0.5],
            "q_high_bruto": preds[q_high],
            "q_low_cqr": q_low_cqr,
            "q_high_cqr": q_high_cqr,
        })

        # score de não-conformidade (CQR): o quanto o valor real, se algum dia soubéssemos,
        # ficaria fora do intervalo bruto — usado só para calibrar passos FUTUROS
        scores_conformidade.append(max(preds[q_low] - y_real, y_real - preds[q_high]))

    return pd.DataFrame(registros).set_index("data")


resultado_cdi = backtest_walk_forward_cqr(dados_modelo, COLUNAS_X, "cdi")
resultado_dolar = backtest_walk_forward_cqr(dados_modelo, COLUNAS_X, "dolar")

registrar_log("backtest_cqr", "ok", {"n_cdi": len(resultado_cdi), "n_dolar": len(resultado_dolar)})
resultado_dolar.tail()

/tmp/ipykernel_1940/1978311938.py:32: IterationLimitWarning: Maximum number of iterations (2000) reached.
  resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)


/tmp/ipykernel_1940/1978311938.py:32: IterationLimitWarning: Maximum number of iterations (2000) reached.
  resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)


/tmp/ipykernel_1940/1978311938.py:32: IterationLimitWarning: Maximum number of iterations (2000) reached.
  resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)
/tmp/ipykernel_1940/1978311938.py:32: IterationLimitWarning: Maximum number of iterations (2000) reached.
  resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)


/tmp/ipykernel_1940/1978311938.py:32: IterationLimitWarning: Maximum number of iterations (2000) reached.
  resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)


/tmp/ipykernel_1940/1978311938.py:32: IterationLimitWarning: Maximum number of iterations (2000) reached.
  resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)


2026-09-01 00:53:55,301 | INFO | [backtest_cqr] ok | {'n_cdi': 162, 'n_dolar': 162}


,y_real,q_low_bruto,q_mediana,q_high_bruto,q_low_cqr,q_high_cqr
data,,,,,,
2026-03-01,-3.794477,-2.935845,-1.082058,2.400717,-2.935845,2.400717
2026-04-01,-0.981011,-3.121298,-1.013169,2.499322,-3.121298,2.499322
2026-05-01,2.886840,-3.320212,-0.890918,2.334043,-3.320212,2.334043
2026-06-01,-0.265693,-3.123018,-0.771925,3.173120,-3.123018,3.173120
2026-07-01,0.766806,-3.050526,-0.679695,3.210045,-3.050526,3.210045


## 2.3 Verificação de calibração (validação obrigatória)

O intervalo bruto da regressão quantílica (10/90) é nominalmente de 80%. Ele quase
nunca acerta esse número na prática — é exatamente por isso que existe o conformal.
Comparamos os dois: cobertura empírica do intervalo bruto vs. do intervalo corrigido
pelo CQR. O corrigido precisa ficar perto de 80%; se não ficar, a janela de calibração
ou o modelo base precisam ser revistos antes de qualquer uso real.

In [18]:
def checar_calibracao(resultado_backtest: pd.DataFrame) -> dict:
    """
    Mede a cobertura empírica dos intervalos bruto e corrigido (CQR) sobre o período
    de backtest em que a correção já estava disponível (janela de calibração cheia).
    """
    df = resultado_backtest.dropna(subset=["q_low_cqr", "q_high_cqr"])
    cobertura_bruta = ((df["y_real"] >= df["q_low_bruto"]) & (df["y_real"] <= df["q_high_bruto"])).mean()
    cobertura_cqr = ((df["y_real"] >= df["q_low_cqr"]) & (df["y_real"] <= df["q_high_cqr"])).mean()
    largura_media_bruta = (df["q_high_bruto"] - df["q_low_bruto"]).mean()
    largura_media_cqr = (df["q_high_cqr"] - df["q_low_cqr"]).mean()
    return {
        "n_obs_validas": len(df),
        "cobertura_nominal": 0.80,
        "cobertura_empirica_bruta": round(cobertura_bruta, 3),
        "cobertura_empirica_cqr": round(cobertura_cqr, 3),
        "largura_media_bruta_pp": round(largura_media_bruta, 2),
        "largura_media_cqr_pp": round(largura_media_cqr, 2),
    }


calibracao_cdi = checar_calibracao(resultado_cdi)
calibracao_dolar = checar_calibracao(resultado_dolar)
registrar_log("calibracao_cqr", "ok", {"cdi": calibracao_cdi, "dolar": calibracao_dolar})

pd.DataFrame({"cdi": calibracao_cdi, "dolar": calibracao_dolar}).T

2026-09-01 00:53:55,320 | INFO | [calibracao_cqr] ok | {'cdi': {'n_obs_validas': 126, 'cobertura_nominal': 0.8, 'cobertura_empirica_bruta': np.float64(0.706), 'cobertura_empirica_cqr': np.float64(0.786), 'largura_media_bruta_pp': np.float64(0.14), 'largura_media_cqr_pp': np.float64(0.16)}, 'dolar': {'n_obs_validas': 126, 'cobertura_nominal': 0.8, 'cobertura_empirica_bruta': np.float64(0.778), 'cobertura_empirica_cqr': np.float64(0.794), 'largura_media_bruta_pp': np.float64(8.16), 'largura_media_cqr_pp': np.float64(8.73)}}


,n_obs_validas,cobertura_nominal,cobertura_empirica_bruta,cobertura_empirica_cqr,largura_media_bruta_pp,largura_media_cqr_pp
cdi,126.0,0.8,0.706,0.786,0.14,0.16
dolar,126.0,0.8,0.778,0.794,8.16,8.73


## 2.4 Cenário atual → distribuição de retorno acumulado em 12 meses (bootstrap em blocos)

A regressão quantílica walk-forward responde "mês que vem". Para uso na mesa importa
mais o acumulado de 12 meses. Em vez de assumir independência mês a mês (o que
subestimaria a incerteza), encadeamos blocos contíguos de resíduos históricos — isso
preserva a autocorrelação de curto prazo da série — somados à mediana condicional do
cenário atual, e compomos o retorno resultante.

In [19]:
def bootstrap_blocos_retorno_acumulado(
    residuos: np.ndarray,
    mediana_condicional: float,
    horizonte_meses: int = 12,
    tamanho_bloco: int = 3,
    n_simulacoes: int = 5000,
    seed: int = 42,
) -> np.ndarray:
    """
    Simula `n_simulacoes` trajetórias do retorno acumulado em `horizonte_meses`,
    encadeando blocos contíguos de `residuos` históricos (bootstrap em blocos, preserva
    dependência temporal de curto prazo) e somando à mediana condicional prevista para
    o cenário atual — assume que o cenário (e por isso a mediana) se mantém ao longo do
    horizonte, uma simplificação razoável para uma v1.
    """
    rng = np.random.default_rng(seed)
    n = len(residuos)
    simulacoes = np.empty(n_simulacoes)

    for s in range(n_simulacoes):
        retornos_simulados = []
        while len(retornos_simulados) < horizonte_meses:
            inicio = rng.integers(0, n - tamanho_bloco + 1)
            bloco = residuos[inicio: inicio + tamanho_bloco]
            retornos_simulados.extend((mediana_condicional + bloco).tolist())
        fator_acumulado = np.prod(1 + np.array(retornos_simulados[:horizonte_meses]) / 100)
        simulacoes[s] = (fator_acumulado - 1) * 100

    return simulacoes


def distribuicao_12m_cenario_atual(dados: pd.DataFrame, colunas_x: list[str], coluna_y: str) -> dict:
    """
    Ajusta a regressão quantílica com todo o histórico disponível, extrai os resíduos
    do modelo de mediana, e usa bootstrap em blocos para simular a distribuição do
    retorno acumulado em 12 meses a partir do cenário (Selic/IPCA) mais recente.
    """
    X = sm.add_constant(dados[colunas_x])
    y = dados[coluna_y]

    modelo_mediana = QuantReg(y, X).fit(q=0.5, max_iter=2000)
    residuos = (y - modelo_mediana.predict(X)).to_numpy()

    cenario_atual = X.iloc[[-1]]
    mediana_atual = modelo_mediana.predict(cenario_atual).iloc[0]

    simulacoes = bootstrap_blocos_retorno_acumulado(residuos, mediana_atual)

    return {
        "data_cenario": dados.index[-1],
        "mediana_mensal_condicional": mediana_atual,
        "q10_12m": np.quantile(simulacoes, 0.10),
        "q50_12m": np.quantile(simulacoes, 0.50),
        "q90_12m": np.quantile(simulacoes, 0.90),
    }


distribuicao_cdi_12m = distribuicao_12m_cenario_atual(dados_modelo, COLUNAS_X, "cdi")
distribuicao_dolar_12m = distribuicao_12m_cenario_atual(dados_modelo, COLUNAS_X, "dolar")

registrar_log("distribuicao_12m", "ok", {
    "cdi": {k: (str(v) if k == "data_cenario" else round(v, 2)) for k, v in distribuicao_cdi_12m.items()},
    "dolar": {k: (str(v) if k == "data_cenario" else round(v, 2)) for k, v in distribuicao_dolar_12m.items()},
})

pd.DataFrame({"cdi": distribuicao_cdi_12m, "dolar": distribuicao_dolar_12m}).T

2026-09-01 00:53:55,470 | INFO | [distribuicao_12m] ok | {'cdi': {'data_cenario': '2026-07-01 00:00:00', 'mediana_mensal_condicional': np.float64(1.09), 'q10_12m': np.float64(13.61), 'q50_12m': np.float64(13.88), 'q90_12m': np.float64(14.11)}, 'dolar': {'data_cenario': '2026-07-01 00:00:00', 'mediana_mensal_condicional': np.float64(-0.65), 'q10_12m': np.float64(-20.27), 'q50_12m': np.float64(-6.38), 'q90_12m': np.float64(15.05)}}


,data_cenario,mediana_mensal_condicional,q10_12m,q50_12m,q90_12m
cdi,2026-07-01 00:00:00,1.089905,13.61224,13.875499,14.111014
dolar,2026-07-01 00:00:00,-0.653202,-20.265397,-6.382662,15.053952


## 2.5 Relatório consolidado por classe de ativo

In [20]:
def relatorio_classe_ativo(nome_classe: str, resultado_backtest: pd.DataFrame, calibracao: dict, distribuicao_12m: dict) -> str:
    """Monta o texto de saída da Fase 2 para uma classe de ativo, no estilo da frase da Fase 1."""
    ultima_previsao = resultado_backtest.dropna(subset=["q_low_cqr", "q_high_cqr"]).iloc[-1]
    return (
        f"[{nome_classe.upper()}] Cenário de {distribuicao_12m['data_cenario'].strftime('%m/%Y')}: "
        f"retorno esperado no próximo mês entre {ultima_previsao['q_low_cqr']:.2f}% e "
        f"{ultima_previsao['q_high_cqr']:.2f}% (80% de confiança, calibração histórica "
        f"empírica de {calibracao['cobertura_empirica_cqr']*100:.0f}%). "
        f"Acumulado em 12 meses: entre {distribuicao_12m['q10_12m']:.2f}% e "
        f"{distribuicao_12m['q90_12m']:.2f}%, mediana {distribuicao_12m['q50_12m']:.2f}% "
        f"(bootstrap em blocos, cenário atual mantido constante)."
    )


print(relatorio_classe_ativo("CDI", resultado_cdi, calibracao_cdi, distribuicao_cdi_12m))
print(relatorio_classe_ativo("Dólar", resultado_dolar, calibracao_dolar, distribuicao_dolar_12m))

[CDI] Cenário de 07/2026: retorno esperado no próximo mês entre 0.97% e 1.19% (80% de confiança, calibração histórica empírica de 79%). Acumulado em 12 meses: entre 13.61% e 14.11%, mediana 13.88% (bootstrap em blocos, cenário atual mantido constante).
[DÓLAR] Cenário de 07/2026: retorno esperado no próximo mês entre -3.05% e 3.21% (80% de confiança, calibração histórica empírica de 79%). Acumulado em 12 meses: entre -20.27% e 15.05%, mediana -6.38% (bootstrap em blocos, cenário atual mantido constante).
